# KNN (K-Nearest Neighbors) — Classification

A practical intermediate-level notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
df = pd.read_csv("knn_dataset.csv")
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
# Change this to the actual target column in the dataset
target_column = "target"

X = df.drop(columns=[target_column])
y = df[target_column]

X.head()

In [ ]:
y.value_counts()

In [ ]:
X = X.select_dtypes(include=np.number)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

In [ ]:
y_pred = knn.predict(X_test_scaled)

print('Accuracy:', accuracy_score(y_test, y_pred))

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("KNN Confusion Matrix")
plt.show()

In [ ]:
k_values = range(1, 16)
accuracy_scores = []

for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_scaled, y_train)
    pred = model.predict(X_test_scaled)
    accuracy_scores.append(accuracy_score(y_test, pred))

plt.figure(figsize=(8, 4))
plt.plot(k_values, accuracy_scores, marker="o")
plt.xlabel("K")
plt.ylabel("Accuracy")
plt.title("Accuracy vs K")
plt.xticks(list(k_values))
plt.grid(True)
plt.show()

In [ ]:
param_grid = {
    "n_neighbors": range(1, 16),
    "weights": ["uniform", "distance"]
}

grid = GridSearchCV(
    KNeighborsClassifier(),
    param_grid,
    cv=5,
    scoring="accuracy"
)

grid.fit(X_train_scaled, y_train)

print("Best Parameters:", grid.best_params_)
print("Best CV Accuracy:", grid.best_score_)

In [ ]:
best_knn = grid.best_estimator_
final_pred = best_knn.predict(X_test_scaled)

print('Final Test Accuracy:', accuracy_score(y_test, final_pred))
print()
print(classification_report(y_test, final_pred))

In [ ]:
final_cm = confusion_matrix(y_test, final_pred)

sns.heatmap(final_cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Final KNN Confusion Matrix")
plt.show()

In [ ]:
# Replace these example values with one new observation from the dataset features.
new_data = pd.DataFrame([[0, 0]], columns=X.columns)

new_data_scaled = scaler.transform(new_data)
prediction = best_knn.predict(new_data_scaled)

print("Predicted Class:", prediction[0])